In [1]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities import extractor
import uproot
import awkward as ak    

x_MH200=extractor("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH200_LH_2017.root", "Events")


file=uproot.open("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH200_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
booleanas=tree.arrays(["FatJet_isMatchedWith2BHadrons"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]
Fatjet_isMatchedWith2BHadrons= booleanas["FatJet_isMatchedWith2BHadrons"]

#Filtriamo i dati

mask = (ak.flatten(Fatjet_isMatchedWithA) == 1) & (ak.flatten(Fatjet_isMatchedWith2BHadrons) == 1)
x_filtered = x_MH200[mask]


/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /home/riccardo/anaconda3/envs/rootnev/include/site/python3.14); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "
/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


In [2]:
from scipy.special import voigt_profile
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()

x_easy=[x for x in x_plot if 150 < x < 250]

def voigt(x, norm, mu, sigma, gamma):
    return voigt_profile(x-mu, sigma, gamma) * norm

bin_counts, bin_edges = np.histogram(x_easy, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_easy) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_easy) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt)

m_voigt=Minuit(ls_voigt,  norm=1, mu=200, sigma=5, gamma=1)
m_voigt.limits["mu"]= (175, 225)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.migrad()


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 1996 (χ²/ndof = 43.4)      │              Nfcn = 147              │
│ EDM = 5.15e-05 (Goal: 0.0002)    │            time = 0.2 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm  │   0.980   │   0.006   │            │            │         │         │       │
│ 1 │ mu    │  200.93   │   0.10    │            │            │   175   │   225   │       │
│ 2 │ sigma │   14.06   │   0.20    │            │            │   0.1   │   20    │       │
│ 3 │ gamma │   2.37    │   0.21    │            │            │  0.01   │   10    │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬─────────────────────────────────────────┐
│       │      norm        mu     sigma     gamma │
├───────┼─────────────────────────────────────────┤
│  norm │  3.54e-05  0.047e-3 -0.502e-3  0.615e-3 │
│    mu │  0.047e-3    0.0111    -0.008     0.004 │
│ sigma │ -0.502e-3    -0.008    0.0392     -0.04 │
│ gamma │  0.615e-3     0.004     -0.04    0.0457 │
└───────┴─────────────────────────────────────────┘

In [3]:
fit_MH200_values={}
fit_MH200_errors={}

fit_values={'MH200': fit_MH200_values,}
fit_errors={'MH200_errors': fit_MH200_errors}

for param in m_voigt.parameters:
    fit_MH200_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deveestarre gli errori 
    fit_MH200_errors[error] = m_voigt.errors[error]

print(fit_MH200_values)
print(fit_MH200_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH200"]=fit_MH200_values
results["MH200_errors"]=fit_MH200_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH200"]=fit_MH200_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH200_errors"]=fit_MH200_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  


{'norm': 0.9800552122194244, 'mu': 200.93117749041895, 'sigma': 14.060459250151368, 'gamma': 2.3731151528940337}
{'norm': 0.0059496891702595055, 'mu': 0.10536236689462442, 'sigma': 0.19789115032979243, 'gamma': 0.21364182420778066}


In [6]:
from Utilities import cb_pdf

x_CB=x_easy
y_CB=[cb_pdf(xi, mu=200, sigma=10, beta=1.5, m=30 ) for xi in x_CB]

ls_CB=LeastSquares(bin_centers, bin_densities, yerr, model=cb_pdf)
m=Minuit(ls_CB, mu=200, sigma=10, beta=1.5, m=30)
m.limits["mu"]= (150, 250)
m.limits["sigma"]= (1, 20)
m.limits["beta"]= (0.1, 10)
#m.limits["m"]= (0.1, 100)
m.migrad()  





/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/scipy/stats/_continuous_distns.py:11686: RuntimeWarning: overflow encountered in power
  return ((m/beta)**m * np.exp(-beta**2 / 2.0) *
/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/scipy/stats/_continuous_distns.py:11686: RuntimeWarning: invalid value encountered in multiply
  return ((m/beta)**m * np.exp(-beta**2 / 2.0) *


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 172.1 (χ²/ndof = 3.7)      │             Nfcn = 3400              │
│ EDM = 0.62 (Goal: 0.0002)        │            time = 0.4 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│         INVALID Minimum          │   ABOVE EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ mu    │  204.48   │   0.15    │            │            │   150   │   250   │       │
│ 1 │ sigma │   12.87   │   0.12    │            │            │    1    │   20    │       │
│ 2 │ beta  │   0.688   │   0.013   │            │            │   0.1   │   10    │       │
│ 3 │ m     │  0.13e3   │  0.10e3   │            │            │         │         │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬─────────────────────────────────────────────┐
│       │         mu      sigma       beta          m │
├───────┼─────────────────────────────────────────────┤
│    mu │     0.0238     -0.014   -1.58e-3      1.616 │
│ sigma │     -0.014      0.014    1.19e-3     -0.869 │
│  beta │   -1.58e-3    1.19e-3   0.000178 -460.61e-3 │
│     m │      1.616     -0.869 -460.61e-3   1.03e+04 │
└───────┴─────────────────────────────────────────────┘

In [8]:
from numba_stats import truncnorm

def gaussiana(x, mu, sigma):
    return truncnorm.pdf(x, *x_range, loc=mu, scale=sigma)

x_gauss=x_easy
x_range=(0, max(x_gauss))

bin_counts, bin_edges = np.histogram(x_easy, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_plot) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_plot) * bin_width) # Errore standard per i dati binned

ls_gauss=LeastSquares(bin_centers, bin_densities, yerr, model=gaussiana)
m_gauss=Minuit(ls_gauss, mu=200, sigma=10)
m_gauss.limits["mu"]= (150, 250)
m_gauss.limits["sigma"]= (1, 20)
m_gauss.migrad()



┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 2612 (χ²/ndof = 54.4)      │              Nfcn = 52               │
│ EDM = 1.54e-08 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ mu    │  200.64   │   0.09    │            │            │   150   │   250   │       │
│ 1 │ sigma │   16.05   │   0.07    │            │            │    1    │   20    │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬───────────────┐
│       │     mu  sigma │
├───────┼───────────────┤
│    mu │ 0.0085 -0.004 │
│ sigma │ -0.004 0.0055 │
└───────┴───────────────┘